# Yale Colon Analysis V

## Marker-pair co-expression scatter plots (PTPRC vs PDGFRA, and many more)

This notebook loads `colon_adata_clustered.h5ad` and generates **2D scatter plots for many marker gene pairs**.

For each gene pair (X vs Y):
- X-axis = gene X expression
- Y-axis = gene Y expression
- Points are colored by category using simple thresholds (defaults: `> 0.0` for each gene):
  - X only
  - Y only
  - Both (double-positive)

Outputs (saved to disk; plots are *not* shown inline by default):
- One PNG per gene pair in `feature_plots/gene_pair_scatters/`
- Per-pair tables of **double-positive cells** in `feature_plots/gene_pair_scatters/double_positive_tables/`
- Combined CSV of all double-positive cells across pairs in `feature_plots/gene_pair_scatters/all_double_positive_cells.csv`

Use-case: track likely “double-marker” cells and later compare against FastReseg/RNA segmentation outputs.


In [1]:
# 1) Import Libraries & Set Plot Style

from __future__ import annotations

from pathlib import Path
import difflib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from scipy import sparse
import anndata as ad
from IPython.display import display


# Plot style tuned for dense scatter plots
mpl.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "figure.figsize": (7.5, 6.5),
        "axes.titlesize": 13,
        "axes.labelsize": 12,
        "legend.fontsize": 10,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

PLOT_DIR = Path("feature_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = Path("colon_adata_clustered.h5ad")
assert DATA_PATH.exists(), f"Missing file: {DATA_PATH.resolve()}"


In [2]:
# 2) Load Single-Cell Expression Data (AnnData)

adata = ad.read_h5ad(DATA_PATH)

print("Loaded:", DATA_PATH)
print("adata:", adata)
print("X shape (cells x genes):", adata.shape)
print("Has .raw?", adata.raw is not None)
print("Layers:", list(adata.layers.keys()) if hasattr(adata, "layers") else [])

# Optional: choose a layer to plot from (set to None to use .X)
# Example: USE_LAYER = "log1p"  (only if such a layer exists)
USE_LAYER: str | None = None

# Optional: prefer adata.raw when available (common for storing normalized/log1p)
PREFER_RAW = True


Loaded: colon_adata_clustered.h5ad
adata: AnnData object with n_obs × n_vars = 424423 × 3000
    obs: 'fov', 'RNA_RNA.QC.Module_Cell.Typing.InSituType.1_1_clusters', 'RNA_RNA.QC.Module_Cell.Typing.InSituType.1_1_posterior_probability', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'Width', 'Height', 'Mean.PanCK', 'Max.PanCK', 'Mean.G', 'Max.G', 'Mean.Membrane', 'Max.Membrane', 'Mean.CD45', 'Max.CD45', 'Mean.DAPI', 'Max.DAPI', 'SplitRatioToLocal', 'NucArea', 'NucAspectRatio', 'Circularity', 'Eccentricity', 'Perimeter', 'Solidity', 'cell_id', 'assay_type', 'version', 'Run_Tissue_name', 'Panel', 'cellSegmentationSetId', 'cellSegmentationSetName', 'slide_ID', 'CenterX_global_px', 'CenterY_global_px', 'cell_ID', 'unassignedTranscripts', 'median_RNA', 'RNA_quantile_0.75', 'RNA_quantile_0.8', 'RNA_quantile_0.85', 'RNA_quantile_0.9', 'RNA_quantile_0.95', 'RNA_quantile_0.99', 'nCount_RNA', 'nFeature_RNA', 'median_negprobes', 'negprobes_quantile_0.75', 'negprobes_quantile_0.8', 

In [7]:
# 3) Marker list + gene validation + extract expression vectors

from itertools import combinations

# Marker genes you provided (plus metadata for optional pair filtering)
MARKERS = [
    {"gene": "CDH1", "expressed_in": "epithelial", "also_known_as": "E-cadherin"},
    {"gene": "EPCAM", "expressed_in": "epithelial", "also_known_as": ""},
    {"gene": "PTPRC", "expressed_in": "pan-immune", "also_known_as": "CD45"},
    {"gene": "MS4A1", "expressed_in": "B-cells", "also_known_as": "CD20"},
    {"gene": "CD19", "expressed_in": "B-cells", "also_known_as": ""},
    {"gene": "MZB1", "expressed_in": "plasma cells", "also_known_as": ""},
    {"gene": "CCR10", "expressed_in": "plasma cells", "also_known_as": ""},
    {"gene": "SDC1", "expressed_in": "plasma cells", "also_known_as": "syndecan1"},
    {"gene": "CD3E", "expressed_in": "T-cells", "also_known_as": ""},
    {"gene": "PECAM1", "expressed_in": "Endothelial", "also_known_as": ""},
    {"gene": "MYH11", "expressed_in": "Smooth muscle cells (SMCs)", "also_known_as": ""},
    {"gene": "ACTG2", "expressed_in": "Smooth muscle cells (SMCs)", "also_known_as": ""},
    {"gene": "MYOCD", "expressed_in": "Smooth muscle cells (SMCs)", "also_known_as": "myocardin"},
    {"gene": "PDGFRA", "expressed_in": "fibroblasts", "also_known_as": ""},
    {"gene": "SPARC", "expressed_in": "fibroblasts", "also_known_as": ""},
    {"gene": "DCN", "expressed_in": "fibroblasts", "also_known_as": "decorin"},
    {"gene": "LUM", "expressed_in": "fibroblasts", "also_known_as": "lumican"},
    {"gene": "COL1A1", "expressed_in": "fibroblasts", "also_known_as": ""},
]

markers_df = pd.DataFrame(MARKERS)
display(markers_df)


def _closest_gene_matches(var_names: pd.Index, gene: str, n: int = 10) -> list[str]:
    candidates = list(map(str, var_names[:]))
    return difflib.get_close_matches(gene, candidates, n=n, cutoff=0.0)


def _resolve_gene(var_names: pd.Index, gene: str) -> str | None:
    if gene in var_names:
        return gene

    # case-insensitive fallback
    upper_map = {str(g).upper(): str(g) for g in var_names}
    if gene.upper() in upper_map:
        return upper_map[gene.upper()]

    return None


def _get_expr_matrix(
    adata: ad.AnnData,
    *,
    use_layer: str | None,
    prefer_raw: bool,
) -> tuple[object, pd.Index, str]:
    """Return (matrix, var_names, matrix_label) where matrix is cells x genes."""

    if prefer_raw and adata.raw is not None:
        return adata.raw.X, adata.raw.var_names, "adata.raw.X"

    if use_layer is not None:
        if use_layer not in adata.layers:
            raise KeyError(f"Layer '{use_layer}' not found. Available: {list(adata.layers.keys())}")
        return adata.layers[use_layer], adata.var_names, f"adata.layers['{use_layer}']"

    return adata.X, adata.var_names, "adata.X"


def _dense_expr_for_genes(
    matrix: object,
    var_names: pd.Index,
    genes: list[str],
) -> tuple[np.ndarray, list[str]]:
    """Return dense float32 expression matrix (cells x len(genes)) and resolved genes."""

    resolved: list[str] = []
    indices: list[int] = []

    for g in genes:
        rg = _resolve_gene(var_names, g)
        if rg is None:
            print(f"Missing gene: {g}. Close matches: {_closest_gene_matches(var_names, g, n=8)[:8]}")
            continue
        idx = int(np.where(var_names == rg)[0][0])
        resolved.append(rg)
        indices.append(idx)

    if len(indices) == 0:
        raise ValueError("None of the requested genes were found in var_names.")

    if sparse.issparse(matrix):
        sub = matrix[:, indices].toarray()
    else:
        sub = np.asarray(matrix[:, indices])

    return sub.astype(np.float32, copy=False), resolved


matrix, var_names, MATRIX_LABEL = _get_expr_matrix(adata, use_layer=USE_LAYER, prefer_raw=PREFER_RAW)

requested_genes = [d["gene"] for d in MARKERS]
expr_mat, resolved_genes = _dense_expr_for_genes(matrix, var_names, requested_genes)

print("Expression source:", MATRIX_LABEL)
print(f"Resolved {len(resolved_genes)}/{len(requested_genes)} requested genes")
print("Resolved genes:", resolved_genes)

# Convenience lookups
cell_ids = adata.obs_names.astype(str).to_numpy()
gene_to_col = {g: i for i, g in enumerate(resolved_genes)}
gene_to_celltype = {row["gene"]: row["expressed_in"] for row in MARKERS if row["gene"] in gene_to_col}


,gene,expressed_in,also_known_as
0,CDH1,epithelial,E-cadherin
1,EPCAM,epithelial,
2,PTPRC,pan-immune,CD45
3,MS4A1,B-cells,CD20
4,CD19,B-cells,
5,MZB1,plasma cells,
6,CCR10,plasma cells,
7,SDC1,plasma cells,syndecan1
8,CD3E,T-cells,
9,PECAM1,Endothelial,


Expression source: adata.raw.X
Resolved 18/18 requested genes
Resolved genes: ['CDH1', 'EPCAM', 'PTPRC', 'MS4A1', 'CD19', 'MZB1', 'CCR10', 'SDC1', 'CD3E', 'PECAM1', 'MYH11', 'ACTG2', 'MYOCD', 'PDGFRA', 'SPARC', 'DCN', 'LUM', 'COL1A1']


In [8]:
# 4) Define thresholds + choose which gene pairs to plot

# Threshold definition:
# - If the expression matrix is log1p-normalized, `> 0` is commonly used.
# - If you are using raw counts, you may prefer `>= 1`.
DEFAULT_THRESH = 0.0
THRESHOLDS = {g: DEFAULT_THRESH for g in resolved_genes}

# Pair selection:
# - If True: only plot pairs from different "expressed_in" groups (useful to find doublets / mixed markers)
# - If False: plot all pairs from the gene list
DIFFERENT_CELLTYPES_ONLY = True

# Limit for quick testing (set to None for all pairs)
MAX_PAIRS: int | None = None


def _pair_list(genes: list[str]) -> list[tuple[str, str]]:
    pairs: list[tuple[str, str]] = []
    for a, b in combinations(genes, 2):
        if DIFFERENT_CELLTYPES_ONLY:
            if gene_to_celltype.get(a) == gene_to_celltype.get(b):
                continue
        pairs.append((a, b))

    if MAX_PAIRS is not None:
        pairs = pairs[:MAX_PAIRS]

    return pairs


pairs = _pair_list(resolved_genes)
print(f"Pairs to plot: {len(pairs):,} (different_celltypes_only={DIFFERENT_CELLTYPES_ONLY}, max_pairs={MAX_PAIRS})")

# Quick peek
pairs[:10]


Pairs to plot: 135 (different_celltypes_only=True, max_pairs=None)


[('CDH1', 'PTPRC'),
 ('CDH1', 'MS4A1'),
 ('CDH1', 'CD19'),
 ('CDH1', 'MZB1'),
 ('CDH1', 'CCR10'),
 ('CDH1', 'SDC1'),
 ('CDH1', 'CD3E'),
 ('CDH1', 'PECAM1'),
 ('CDH1', 'MYH11'),
 ('CDH1', 'ACTG2')]

In [9]:
# 5) Generate scatter plots (saved to disk; not shown)

OUT_DIR = PLOT_DIR / "gene_pair_scatters"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DOUBLE_POS_DIR = OUT_DIR / "double_positive_tables"
DOUBLE_POS_DIR.mkdir(parents=True, exist_ok=True)

SHOW_PLOTS = False  # keep False to avoid rendering hundreds of figures

# Colors (make 'Both' stand out)
COLORS = {
    "X only": "#1f78b4",
    "Y only": "#33a02c",
    "Both": "#e31a1c",
}
DRAW_ORDER = ["X only", "Y only", "Both"]


def _safe_name(s: str) -> str:
    return (
        s.strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(";", "_")
    )


def _plot_one_pair(gx: str, gy: str) -> tuple[pd.DataFrame, dict]:
    """Save plot + per-pair double-positive tables. Return (double_pos_df, counts_dict)."""

    xi = gene_to_col[gx]
    yi = gene_to_col[gy]

    x = expr_mat[:, xi]
    y = expr_mat[:, yi]

    tx = float(THRESHOLDS.get(gx, DEFAULT_THRESH))
    ty = float(THRESHOLDS.get(gy, DEFAULT_THRESH))

    x_pos = x > tx
    y_pos = y > ty

    # Only plot cells expressing at least one gene (matches your request; keeps plots fast/clean)
    plot_mask = x_pos | y_pos

    # Categories (within plot_mask)
    cat = np.full(plot_mask.sum(), "", dtype=object)
    x_only = x_pos[plot_mask] & ~y_pos[plot_mask]
    y_only = ~x_pos[plot_mask] & y_pos[plot_mask]
    both = x_pos[plot_mask] & y_pos[plot_mask]
    cat[x_only] = "X only"
    cat[y_only] = "Y only"
    cat[both] = "Both"

    counts = {
        "pair": f"{gx}__{gy}",
        "gene_x": gx,
        "gene_y": gy,
        "thresh_x": tx,
        "thresh_y": ty,
        "n_total": int(adata.n_obs),
        "n_plot": int(plot_mask.sum()),
        "n_x_only": int(x_only.sum()),
        "n_y_only": int(y_only.sum()),
        "n_both": int(both.sum()),
        "matrix": MATRIX_LABEL,
        "x_celltype": gene_to_celltype.get(gx, ""),
        "y_celltype": gene_to_celltype.get(gy, ""),
    }

    plot_df = pd.DataFrame(
        {
            "cell_id": cell_ids[plot_mask],
            gx: x[plot_mask],
            gy: y[plot_mask],
            "category": cat,
        }
    )

    # --- Plot ---
    fig, ax = plt.subplots()
    for label in DRAW_ORDER:
        m = plot_df["category"] == label
        n = int(m.sum())
        if n == 0:
            continue
        s = 7 if label != "Both" else 10
        alpha = 0.45 if label != "Both" else 0.85
        zorder = 2 if label != "Both" else 3

        ax.scatter(
            plot_df.loc[m, gx],
            plot_df.loc[m, gy],
            s=s,
            alpha=alpha,
            c=COLORS[label],
            edgecolors="none",
            rasterized=True,
            label=f"{label} (n={n:,})",
            zorder=zorder,
        )

    ax.axvline(tx, color="k", lw=1, ls="--", alpha=0.5)
    ax.axhline(ty, color="k", lw=1, ls="--", alpha=0.5)

    ax.set_xlabel(f"{gx} expression")
    ax.set_ylabel(f"{gy} expression")

    ax.set_title(
        f"{gx} vs {gy} (x>{tx:g}, y>{ty:g}; plotted: expressing cells; source: {MATRIX_LABEL})"
    )
    ax.legend(loc="best", frameon=True)
    ax.grid(True, alpha=0.25)

    fig.tight_layout()

    out_png = OUT_DIR / f"{_safe_name(gx)}_vs_{_safe_name(gy)}.png"
    fig.savefig(out_png, bbox_inches="tight")

    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

    # --- Double-positive table ---
    double_pos = plot_df[plot_df["category"] == "Both"].copy()
    if len(double_pos) > 0:
        double_pos["sum"] = double_pos[gx] + double_pos[gy]
        double_pos = double_pos.sort_values("sum", ascending=False)

        # Full list
        out_full = DOUBLE_POS_DIR / f"{_safe_name(gx)}_vs_{_safe_name(gy)}__double_positive.csv"
        double_pos[["cell_id", gx, gy, "category", "sum"]].to_csv(out_full, index=False)

        # Top 10 table (like the example you pasted)
        out_top = DOUBLE_POS_DIR / f"{_safe_name(gx)}_vs_{_safe_name(gy)}__double_positive_top10.csv"
        double_pos[["cell_id", gx, gy, "category", "sum"]].head(10).to_csv(out_top, index=False)

    return double_pos, counts


In [11]:
# 6) Run all pairs + export combined CSVs

pair_counts: list[dict] = []
all_double: list[pd.DataFrame] = []

print("Writing plots to:", OUT_DIR)
print("Writing per-pair double-positive tables to:", DOUBLE_POS_DIR)

for i, (gx, gy) in enumerate(pairs, start=1):
    double_pos, counts = _plot_one_pair(gx, gy)
    pair_counts.append(counts)

    if len(double_pos) > 0:
        # Standardize columns for concatenation
        df = double_pos[["cell_id", gx, gy, "sum"]].copy()
        df = df.rename(columns={gx: "gene_x_expr", gy: "gene_y_expr"})
        df.insert(1, "gene_x", gx)
        df.insert(2, "gene_y", gy)
        df.insert(3, "category", "Both")
        df.insert(4, "matrix", MATRIX_LABEL)
        df.insert(5, "thresh_x", float(THRESHOLDS.get(gx, DEFAULT_THRESH)))
        df.insert(6, "thresh_y", float(THRESHOLDS.get(gy, DEFAULT_THRESH)))
        df.insert(7, "x_celltype", gene_to_celltype.get(gx, ""))
        df.insert(8, "y_celltype", gene_to_celltype.get(gy, ""))
        all_double.append(df)

    if i % 25 == 0 or i == len(pairs):
        print(f"Processed {i:,}/{len(pairs):,} pairs")

# Pair-level counts summary
counts_df = pd.DataFrame(pair_counts)
counts_out = OUT_DIR / "pair_category_counts.csv"
counts_df.to_csv(counts_out, index=False)
print("Saved:", counts_out)

# Combined double-positive table
if len(all_double) > 0:
    all_double_df = pd.concat(all_double, ignore_index=True)
    all_double_out = OUT_DIR / "all_double_positive_cells.csv"
    all_double_df.to_csv(all_double_out, index=False)
    print("Saved:", all_double_out)

    # Optional: per-cell how many pairs they are double-positive for
    cell_pair_counts = (
        all_double_df.groupby("cell_id")["gene_x"].count().rename("n_double_positive_pairs").sort_values(ascending=False)
    )
    cell_pair_counts_out = OUT_DIR / "cell_double_positive_pair_counts.csv"
    cell_pair_counts.to_csv(cell_pair_counts_out)
    print("Saved:", cell_pair_counts_out)

    # Preview a few top rows (by sum) for sanity
    display(all_double_df.sort_values("sum", ascending=False).head(10))
else:
    print("No double-positive cells found across these pairs with the current thresholds.")


Writing plots to: feature_plots/gene_pair_scatters
Writing per-pair double-positive tables to: feature_plots/gene_pair_scatters/double_positive_tables
Processed 25/135 pairs
Processed 50/135 pairs
Processed 75/135 pairs
Processed 100/135 pairs
Processed 125/135 pairs
Processed 135/135 pairs
Saved: feature_plots/gene_pair_scatters/pair_category_counts.csv
Saved: feature_plots/gene_pair_scatters/all_double_positive_cells.csv
Saved: feature_plots/gene_pair_scatters/cell_double_positive_pair_counts.csv


,cell_id,gene_x,gene_y,category,matrix,thresh_x,thresh_y,x_celltype,y_celltype,gene_x_expr,gene_y_expr,sum
96866,c_1_343_168,ACTG2,SPARC,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,5.993961,6.685861,12.679823
101021,c_1_270_1158,MYOCD,COL1A1,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,5.687374,6.937714,12.625088
94233,c_1_171_527,MYH11,COL1A1,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,6.908755,5.525453,12.434208
94234,c_1_25_1611,MYH11,COL1A1,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,6.034684,6.321767,12.356451
94235,c_1_324_155,MYH11,COL1A1,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,5.208492,6.995683,12.204175
66331,c_1_187_149,MZB1,LUM,Both,adata.raw.X,0.0,0.0,plasma cells,fibroblasts,6.099072,6.099072,12.198145
98390,c_1_281_386,ACTG2,COL1A1,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,5.190573,6.977681,12.168255
94236,c_1_93_691,MYH11,COL1A1,Both,adata.raw.X,0.0,0.0,Smooth muscle cells (SMCs),fibroblasts,5.863282,6.267799,12.131082
40999,c_1_303_972,EPCAM,COL1A1,Both,adata.raw.X,0.0,0.0,epithelial,fibroblasts,5.430541,6.526228,11.956770
50581,c_1_270_1158,PTPRC,COL1A1,Both,adata.raw.X,0.0,0.0,pan-immune,fibroblasts,4.997610,6.937714,11.935324


## FastReseg overlap check (double-positive cells)

This section joins the **double-positive cells** detected in this notebook to **FastReseg** outputs.

Key idea:
- FastReseg provides two complementary “what changed?” views:
  - **Cell-level rename summary** (`all_fov_cell_rename_summary.csv`): per old-cell, how many transcripts were moved and what percent.
  - **Transcript-level changes** (`all_fov_changed_transcripts.csv`): per transcript, which gene moved from which cell to which new cell.

What it produces (in a NEW dedicated folder, not inside the gene-pair folder):
- `FastReseg_double_positive_overlap/` (tables + plots)
  - One-row-per-cell table with FastReseg metrics.
  - Affected-only subset.
  - Per-FOV summaries.

Robustness:
- If some FastReseg outputs are missing (failed FOVs), the code skips them.
- If there are multiple combined output folders (`02_combined_outputs`, `_1`, `_2`, `_3`), it uses **all that exist**.


In [1]:
# 7) Load double-positive list + load FastReseg outputs (robust to missing/failed FOVs)

from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ROOT = Path(".")


def _find_double_positive_csv(root: Path) -> Path:
    """Find the combined double-positive CSV regardless of which plot root was used."""
    candidates = [
        root / "feature_plots_tummor" / "gene_pair_scatters" / "all_double_positive_cells.csv",
        root / "feature_plots" / "gene_pair_scatters" / "all_double_positive_cells.csv",
    ]

    for p in candidates:
        if p.exists():
            return p

    tried = "\n".join([f"- {p.resolve()}" for p in candidates])
    raise FileNotFoundError(f"Missing double-positive CSV. Tried:\n{tried}")


# Input from the gene-pair analysis section (already generated earlier)
DOUBLE_POS_ALL_PATH = _find_double_positive_csv(ROOT)
print("Using double-positive CSV:", DOUBLE_POS_ALL_PATH)

# FastReseg roots
FASTRESEG_ROOT = ROOT / "FastReseg_allFOV_same_workflow"
FASTRESEG_PER_FOV_ROOT = FASTRESEG_ROOT / "01_per_fov"

# NEW dedicated output folder (nothing goes into gene-pair folders)
OUT_ROOT = ROOT / "FastReseg_double_positive_overlap"
OUT_TABLES = OUT_ROOT / "tables"
OUT_PLOTS = OUT_ROOT / "plots"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PLOTS.mkdir(parents=True, exist_ok=True)

OUT_CSV = OUT_TABLES / "double_positive_cells_with_fastreseg_metrics.csv"
OUT_AFFECTED_CSV = OUT_TABLES / "double_positive_cells_affected_by_fastreseg.csv"
OUT_FOV_SUMMARY_CSV = OUT_TABLES / "per_fov_double_positive_fastreseg_summary.csv"


def _parse_fov(cell_id: str) -> int | None:
    # Expected format: c_1_<FOV>_<cell>
    m = re.match(r"^c_1_(\d+)_", str(cell_id))
    return int(m.group(1)) if m else None


def _combined_output_dirs() -> list[Path]:
    """Return all existing combined output folders (02_combined_outputs, _1, _2, _3, ...)."""
    if not FASTRESEG_ROOT.exists():
        return []

    dirs = sorted(FASTRESEG_ROOT.glob("02_combined_outputs*"))
    return [d for d in dirs if d.is_dir()]


def _load_any_csv(paths: list[Path]) -> tuple[pd.DataFrame, list[Path]]:
    """Read and concat existing CSVs from paths. Returns (df, used_paths)."""
    frames: list[pd.DataFrame] = []
    used: list[Path] = []

    for p in paths:
        if not p.exists():
            continue
        try:
            frames.append(pd.read_csv(p))
            used.append(p)
        except Exception as e:
            print(f"Skipping unreadable CSV: {p} ({type(e).__name__}: {e})")
            continue

    if len(frames) == 0:
        return pd.DataFrame(), []

    df = pd.concat(frames, ignore_index=True)
    return df, used


# 7A) Load the per-pair double-positive events (may have many rows per cell)
doubles = pd.read_csv(DOUBLE_POS_ALL_PATH)
doubles["fov"] = doubles["cell_id"].map(_parse_fov).astype("Int64")

if doubles["fov"].isna().any():
    bad = doubles.loc[doubles["fov"].isna(), "cell_id"].astype(str).head(10).tolist()
    raise ValueError(f"Could not parse FOV from some cell_id values (examples): {bad}")

# Aggregate to ONE row per cell_id (one cell can be double-positive for multiple pairs)
doubles_cell = (
    doubles.assign(pair=lambda d: d["gene_x"].astype(str) + "__" + d["gene_y"].astype(str))
    .groupby(["cell_id", "fov"], as_index=False)
    .agg(
        n_double_positive_pairs=("pair", "nunique"),
        max_sum_across_pairs=("sum", "max"),
    )
)

print("Double-positive cells (unique):", len(doubles_cell))
print("FOV range:", int(doubles_cell["fov"].min()), "to", int(doubles_cell["fov"].max()))

# 7B) Locate combined output dirs + candidate FastReseg CSVs
combined_dirs = _combined_output_dirs()
print("FastReseg combined output dirs found:")
for d in combined_dirs:
    print("-", d)

rename_paths = [d / "all_fov_cell_rename_summary.csv" for d in combined_dirs]
changed_tx_paths = [d / "all_fov_changed_transcripts.csv" for d in combined_dirs]

# Load rename summary (cell-level) from ALL combined output dirs that exist
rename_all, used_rename_paths = _load_any_csv(rename_paths)

# If no combined rename summary exists, fall back to scanning per-FOV outputs
if len(used_rename_paths) == 0:
    print("No combined rename summaries found; scanning per-FOV folders under:", FASTRESEG_PER_FOV_ROOT)
    frames: list[pd.DataFrame] = []

    for fov in sorted(doubles_cell["fov"].unique().tolist()):
        fov_file = (
            FASTRESEG_PER_FOV_ROOT
            / f"FOV_{int(fov)}"
            / f"FastReseg_cell_rename_summary_FOV_{int(fov)}.csv"
        )
        if not fov_file.exists():
            continue
        try:
            frames.append(pd.read_csv(fov_file))
        except Exception as e:
            print(f"Skipping unreadable file: {fov_file} ({type(e).__name__}: {e})")
            continue

    if len(frames) == 0:
        raise FileNotFoundError(
            "Could not find any FastReseg rename summary CSVs (combined or per-FOV)."
        )

    rename_all = pd.concat(frames, ignore_index=True)

# Validate columns
required_cols = {
    "old_cell",
    "new_cell",
    "n_changed_transcripts",
    "old_cell_total_transcripts",
    "pct_of_old_cell_transcripts",
}
missing_cols = required_cols - set(rename_all.columns)
if missing_cols:
    raise KeyError(f"FastReseg rename summary missing expected columns: {sorted(missing_cols)}")

# Normalize & deduplicate rename summary to one row per old_cell
rename_all = rename_all.copy()
rename_all["old_cell"] = rename_all["old_cell"].astype(str)
rename_all["new_cell"] = rename_all["new_cell"].astype(str)
rename_all["pct_of_old_cell_transcripts"] = pd.to_numeric(
    rename_all["pct_of_old_cell_transcripts"], errors="coerce"
)
rename_all["n_changed_transcripts"] = pd.to_numeric(
    rename_all["n_changed_transcripts"], errors="coerce"
)

rename_dedup = (
    rename_all.sort_values(
        ["pct_of_old_cell_transcripts", "n_changed_transcripts"], ascending=[False, False]
    )
    .drop_duplicates(subset=["old_cell"], keep="first")
)

print("FastReseg changed cells (unique old_cell):", rename_dedup["old_cell"].nunique())

# We will use transcript-level changes later (if the combined CSV exists)
existing_changed_tx_paths = [p for p in changed_tx_paths if p.exists()]
print("FastReseg changed-transcripts files found:")
for p in existing_changed_tx_paths:
    print("-", p)


Using double-positive CSV: feature_plots_tummor/gene_pair_scatters/all_double_positive_cells.csv
Double-positive cells (unique): 52935
FOV range: 1 to 361
FastReseg combined output dirs found:
- FastReseg_allFOV_same_workflow/02_combined_outputs_1
- FastReseg_allFOV_same_workflow/02_combined_outputs_2
- FastReseg_allFOV_same_workflow/02_combined_outputs_3
FastReseg changed cells (unique old_cell): 2188
FastReseg changed-transcripts files found:
- FastReseg_allFOV_same_workflow/02_combined_outputs_1/all_fov_changed_transcripts.csv
- FastReseg_allFOV_same_workflow/02_combined_outputs_2/all_fov_changed_transcripts.csv
- FastReseg_allFOV_same_workflow/02_combined_outputs_3/all_fov_changed_transcripts.csv


In [2]:
# 8) Join double-positive cells -> FastReseg cell-level metrics (rename summary)

out = doubles_cell.merge(
    rename_dedup[
        [
            "old_cell",
            "new_cell",
            "n_changed_transcripts",
            "old_cell_total_transcripts",
            "pct_of_old_cell_transcripts",
        ]
    ],
    left_on="cell_id",
    right_on="old_cell",
    how="left",
)

# Cells missing from rename summary are treated as unchanged
out["fastreseg_changed"] = ~out["old_cell"].isna()
out["fastreseg_new_cell"] = out["new_cell"].fillna(out["cell_id"]).astype(str)

out["fastreseg_n_changed_transcripts_rename"] = (
    pd.to_numeric(out["n_changed_transcripts"], errors="coerce").fillna(0).astype(int)
)
out["fastreseg_old_cell_total_transcripts_rename"] = (
    pd.to_numeric(out["old_cell_total_transcripts"], errors="coerce").fillna(0).astype(int)
)
out["fastreseg_pct_changed_rename"] = (
    pd.to_numeric(out["pct_of_old_cell_transcripts"], errors="coerce").fillna(0.0).astype(float)
)

out = out.drop(
    columns=[
        "old_cell",
        "new_cell",
        "n_changed_transcripts",
        "old_cell_total_transcripts",
        "pct_of_old_cell_transcripts",
    ]
)

affected_cells = set(out.loc[out["fastreseg_changed"], "cell_id"].astype(str).unique())
print("Double-positive cells:", len(out))
print("Double-positive cells changed by FastReseg (rename summary):", len(affected_cells))


Double-positive cells: 52935
Double-positive cells changed by FastReseg (rename summary): 245


In [3]:
# 9) Summarize transcript-level changes for affected double-positive cells + write outputs

# If transcript-level combined files exist, summarize them for the *affected* cells only.
# This avoids loading huge files into memory and stays robust if some combined outputs are missing.

def _summarize_changed_transcripts(
    changed_paths: list[Path],
    cells_of_interest: set[str],
    *,
    chunksize: int = 200_000,
) -> pd.DataFrame:
    if len(changed_paths) == 0 or len(cells_of_interest) == 0:
        return pd.DataFrame(
            columns=[
                "cell_id",
                "fastreseg_changed_tx_n",
                "fastreseg_changed_genes_n",
                "fastreseg_top_changed_genes",
                "fastreseg_top_new_cells",
            ]
        )

    hits: list[pd.DataFrame] = []

    for path in changed_paths:
        if not path.exists():
            continue

        try:
            for chunk in pd.read_csv(path, chunksize=chunksize):
                if "old_cell" not in chunk.columns:
                    raise KeyError("Missing column 'old_cell'")

                chunk["old_cell"] = chunk["old_cell"].astype(str)
                sub = chunk[chunk["old_cell"].isin(cells_of_interest)]
                if len(sub) == 0:
                    continue

                # Keep columns that uniquely define a transcript row (if available)
                keep = [c for c in ["fov", "target", "old_cell", "new_cell", "x", "y", "z"] if c in sub.columns]
                sub = sub[keep].copy()
                hits.append(sub)
        except Exception as e:
            print(f"Skipping unreadable changed-transcripts file: {path} ({type(e).__name__}: {e})")
            continue

    if len(hits) == 0:
        return pd.DataFrame(
            columns=[
                "cell_id",
                "fastreseg_changed_tx_n",
                "fastreseg_changed_genes_n",
                "fastreseg_top_changed_genes",
                "fastreseg_top_new_cells",
            ]
        )

    changed = pd.concat(hits, ignore_index=True)

    # Deduplicate across multiple combined output dirs.
    # Using fov+gene+old/new+coordinates when present prevents collapsing distinct transcripts.
    dedup_cols = [c for c in ["fov", "target", "old_cell", "new_cell", "x", "y", "z"] if c in changed.columns]
    changed = changed.drop_duplicates(subset=dedup_cols)

    # Per-cell summary
    def _top_k_counts(series: pd.Series, k: int = 5) -> str:
        vc = series.astype(str).value_counts().head(k)
        return "; ".join([f"{idx}:{int(val)}" for idx, val in vc.items()])

    # Some files may omit 'new_cell' (rare) or 'target'
    has_target = "target" in changed.columns
    has_new_cell = "new_cell" in changed.columns

    grp = changed.groupby("old_cell")

    out = pd.DataFrame({"cell_id": grp.size().index.astype(str)})
    out["fastreseg_changed_tx_n"] = grp.size().to_numpy(dtype=int)

    if has_target:
        out["fastreseg_changed_genes_n"] = grp["target"].nunique().to_numpy(dtype=int)
        out["fastreseg_top_changed_genes"] = grp["target"].apply(_top_k_counts).to_numpy(dtype=object)
    else:
        out["fastreseg_changed_genes_n"] = 0
        out["fastreseg_top_changed_genes"] = ""

    if has_new_cell:
        out["fastreseg_top_new_cells"] = grp["new_cell"].apply(_top_k_counts).to_numpy(dtype=object)
    else:
        out["fastreseg_top_new_cells"] = ""

    return out


changed_tx_summary = _summarize_changed_transcripts(
    existing_changed_tx_paths,
    affected_cells,
)

# Merge transcript-level summary onto out (fill missing with 0/empty)
out2 = out.merge(changed_tx_summary, on="cell_id", how="left")
out2["fastreseg_changed_tx_n"] = (
    pd.to_numeric(out2["fastreseg_changed_tx_n"], errors="coerce").fillna(0).astype(int)
)
out2["fastreseg_changed_genes_n"] = (
    pd.to_numeric(out2["fastreseg_changed_genes_n"], errors="coerce").fillna(0).astype(int)
)
out2["fastreseg_top_changed_genes"] = out2["fastreseg_top_changed_genes"].fillna("")
out2["fastreseg_top_new_cells"] = out2["fastreseg_top_new_cells"].fillna("")

# Per-FOV summary (how many double-positive cells were affected)
per_fov = (
    out2.groupby("fov")
    .agg(
        n_double_positive_cells=("cell_id", "nunique"),
        n_affected_double_positive_cells=("fastreseg_changed", "sum"),
        median_pct_changed_rename=("fastreseg_pct_changed_rename", "median"),
        max_pct_changed_rename=("fastreseg_pct_changed_rename", "max"),
    )
    .reset_index()
    .sort_values("n_affected_double_positive_cells", ascending=False)
)

# Write outputs
out2 = out2.sort_values(
    ["fastreseg_changed", "fastreseg_pct_changed_rename", "n_double_positive_pairs"],
    ascending=[False, False, False],
)

affected2 = out2[out2["fastreseg_changed"]].copy()

out2.to_csv(OUT_CSV, index=False)
affected2.to_csv(OUT_AFFECTED_CSV, index=False)
per_fov.to_csv(OUT_FOV_SUMMARY_CSV, index=False)

print("Saved:", OUT_CSV)
print("Saved:", OUT_AFFECTED_CSV)
print("Saved:", OUT_FOV_SUMMARY_CSV)

print("--- Summary ---")
print("Double-positive cells:", len(out2))
print("Affected by FastReseg (rename summary):", int(out2["fastreseg_changed"].sum()))
if len(affected2) > 0:
    print("Median % changed (affected only):", float(np.median(affected2["fastreseg_pct_changed_rename"])))
    print("Max % changed (affected only):", float(np.max(affected2["fastreseg_pct_changed_rename"])))


Saved: FastReseg_double_positive_overlap/tables/double_positive_cells_with_fastreseg_metrics.csv
Saved: FastReseg_double_positive_overlap/tables/double_positive_cells_affected_by_fastreseg.csv
Saved: FastReseg_double_positive_overlap/tables/per_fov_double_positive_fastreseg_summary.csv
--- Summary ---
Double-positive cells: 52935
Affected by FastReseg (rename summary): 245
Median % changed (affected only): 0.628930817610063
Max % changed (affected only): 100.0


In [4]:
# 10) Visual summaries (saved to disk)

# Reload from disk (ensures you are looking at the saved artifact)
out = pd.read_csv(OUT_CSV)
affected = out[out["fastreseg_changed"]]

# A) Histogram of % changed among affected double-positive cells
fig, ax = plt.subplots(figsize=(7.5, 5.0), dpi=120)

if len(affected) == 0:
    ax.text(
        0.5,
        0.5,
        "No double-positive cells were changed by FastReseg",
        ha="center",
        va="center",
    )
    ax.set_axis_off()
else:
    ax.hist(
        affected["fastreseg_pct_changed_rename"].astype(float),
        bins=40,
        color="#4c78a8",
        alpha=0.9,
        edgecolor="white",
    )
    ax.set_xlabel("FastReseg % of old-cell transcripts changed (rename summary)")
    ax.set_ylabel("Number of double-positive cells")
    ax.set_title("FastReseg impact on double-positive cells")
    ax.grid(True, alpha=0.25)

fig.tight_layout()
out_png = OUT_PLOTS / "fastreseg_pct_changed_histogram_double_positive.png"
fig.savefig(out_png, bbox_inches="tight")
plt.close(fig)
print("Saved:", out_png)

# B) Top FOVs by number of affected double-positive cells
if len(affected) > 0:
    per_fov = (
        affected.groupby("fov")["cell_id"].nunique().sort_values(ascending=False).head(25)
    )

    fig, ax = plt.subplots(figsize=(9.0, 5.5), dpi=120)
    ax.bar(per_fov.index.astype(str), per_fov.values, color="#f58518")
    ax.set_xlabel("FOV")
    ax.set_ylabel("# affected double-positive cells")
    ax.set_title("Top FOVs: FastReseg-changed double-positive cells")
    ax.tick_params(axis="x", labelrotation=60)
    ax.grid(True, axis="y", alpha=0.25)
    fig.tight_layout()

    out_png = OUT_PLOTS / "top_fovs_affected_double_positive_bar.png"
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)

# C) Relationship: how many marker-pairs vs % changed
if len(affected) > 0:
    fig, ax = plt.subplots(figsize=(7.5, 5.0), dpi=120)
    ax.scatter(
        affected["n_double_positive_pairs"].astype(int),
        affected["fastreseg_pct_changed_rename"].astype(float),
        s=18,
        alpha=0.7,
        c="#54a24b",
        edgecolors="none",
        rasterized=True,
    )
    ax.set_xlabel("# marker pairs where cell is double-positive")
    ax.set_ylabel("FastReseg % changed (rename summary)")
    ax.set_title("Double-positive strength vs FastReseg impact")
    ax.grid(True, alpha=0.25)
    fig.tight_layout()

    out_png = OUT_PLOTS / "n_pairs_vs_pct_changed_scatter.png"
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", out_png)


Saved: FastReseg_double_positive_overlap/plots/fastreseg_pct_changed_histogram_double_positive.png
Saved: FastReseg_double_positive_overlap/plots/top_fovs_affected_double_positive_bar.png
Saved: FastReseg_double_positive_overlap/plots/n_pairs_vs_pct_changed_scatter.png
